# CA6: Policy Gradient Methods - Modular Implementation and Analysis

## Abstract

This notebook presents a comprehensive, modular implementation and analysis of various policy gradient methods in deep reinforcement learning. We explore the theoretical foundations, practical implementations, and performance characteristics of algorithms such as REINFORCE, REINFORCE with Baseline, Actor-Critic, and Proximal Policy Optimization (PPO). The implementation emphasizes a clean, `src`-based architecture for maintainability and scalability. Through systematic experimentation and visualization, we demonstrate the convergence properties, sample efficiency, and practical trade-offs of these algorithms on both discrete and continuous control tasks, providing insights into their application in diverse reinforcement learning scenarios.

**Keywords:** Policy Gradient, REINFORCE, Actor-Critic, PPO, Deep Reinforcement Learning, Continuous Control, Variance Reduction.


## 1. Introduction

Policy gradient methods are a class of reinforcement learning algorithms that directly optimize the policy function, which maps states to actions, to maximize the expected return. Unlike value-based methods that learn a value function to implicitly derive a policy, policy gradient methods explicitly parameterize the policy and update its parameters by following the gradient of the expected return. This direct optimization offers several advantages, particularly in environments with continuous action spaces or where learning a stochastic policy is beneficial.

### 1.1 Motivation for Policy Gradient Methods

1.  **Direct Policy Optimization:** Policy gradient methods directly optimize the behavior that an agent learns, which can be more intuitive and robust in complex tasks. This contrasts with value-based methods (e.g., Q-learning, DQN) where a policy is derived indirectly from a learned value function.
2.  **Handling Continuous Action Spaces:** Policy gradient methods naturally extend to continuous action spaces by parameterizing a probability distribution over actions (e.g., Gaussian distribution), from which actions can be sampled. Value-based methods often struggle with continuous actions, requiring discretization or more complex approximations.
3.  **Learning Stochastic Policies:** In environments where an optimal deterministic policy might not exist (e.g., poker, environments with unobserved states), learning a stochastic policy is crucial. Policy gradient methods can inherently learn and represent stochastic policies.
4.  **Smoother Convergence:** By directly optimizing the policy, these methods can sometimes exhibit smoother convergence properties compared to value-based methods, which can suffer from oscillations due to target value fluctuations.

### 1.2 Learning Objectives

Upon completing this assignment, you will be able to:

- Understand the fundamental principles of the Policy Gradient Theorem.
- Implement and analyze the classical REINFORCE algorithm.
- Enhance REINFORCE with a learned baseline to reduce variance.
- Implement the Actor-Critic framework, understanding the interaction between policy (actor) and value function (critic).
- Implement and understand advanced policy gradient algorithms such as Proximal Policy Optimization (PPO).
- Apply policy gradient methods to solve continuous control problems.
- Conduct systematic comparisons and analyses of different policy gradient variants.
- Utilize a modular code structure for building and experimenting with RL algorithms.

### 1.3 Prerequisites

To effectively engage with this notebook and assignment, you should have a solid understanding of:

- **Reinforcement Learning Fundamentals:** Concepts such as states, actions, rewards, episodes, returns, value functions ($V(s)$, $Q(s,a)$), and the Bellman equation.
- **Deep Learning with PyTorch:** Proficiency in building neural networks, defining loss functions, and using optimizers in PyTorch.
- **Probability and Statistics:** Basic understanding of probability distributions, expected values, variance, and gradient ascent/descent.
- **Previous Assignments (CA1-CA5):** Familiarity with concepts covered in earlier assignments on fundamental RL algorithms and deep Q-networks.


## 2. Theoretical Foundations

### 2.1 The Policy Gradient Theorem

The Policy Gradient Theorem provides a fundamental result that allows us to compute the gradient of the expected return with respect to the policy parameters, even without knowing the environment's dynamics. For a parameterized policy $\pi_\theta(a|s)$, the objective is to maximize the expected return $J(\theta) = \mathbb{E}_{\pi_\theta} [G_0]$, where $G_0$ is the total discounted return from the start state.

The policy gradient theorem states that:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) G_t \right]$$

where $G_t$ is the return from time step $t$.

This can also be written using the action-value function $Q^{\pi_\theta}(s,a)$ as:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) Q^{\pi_\theta}(s,a) \right]$$

### 2.2 REINFORCE Algorithm

REINFORCE (also known as Monte Carlo Policy Gradient) is the simplest policy gradient algorithm, directly applying the Policy Gradient Theorem. It works by running an episode to completion, calculating the return $G_t$ for each time step, and then updating the policy parameters $\theta$ in the direction of the estimated gradient:

$$\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a_t|s_t) G_t$$

While simple, REINFORCE can suffer from high variance in its gradient estimates, as the return $G_t$ is highly variable across episodes. This can lead to slow and unstable learning.

### 2.3 Variance Reduction with a Baseline

To address the high variance of REINFORCE, a common technique is to subtract a baseline function $b(s)$ from the return. The most common baseline is the state-value function $V^{\pi_\theta}(s)$. The policy gradient then becomes:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) (G_t - b(s_t)) \right]$$

Subtracting a baseline does not change the expected value of the gradient, but it can significantly reduce its variance, leading to more stable learning. When $b(s_t) = V^{\pi_\theta}(s_t)$, the term $(G_t - V^{\pi_\theta}(s_t))$ becomes an estimate of the advantage function $A^{\pi_\theta}(s_t, a_t)$.

### 2.4 Actor-Critic Methods

Actor-Critic methods combine policy-based and value-based approaches. They learn two functions simultaneously:

- **Actor:** The policy $\pi_\theta(a|s)$, which selects actions.
- **Critic:** The value function $V_\phi(s)$ or $Q_\phi(s,a)$, which estimates the value of states or state-action pairs.

The critic is used to estimate the advantage function, which then guides the actor's policy updates. This allows for bootstrapping, where the value function is updated using TD errors, reducing variance further than Monte Carlo methods like REINFORCE. The actor is updated using the critic's advantage estimate:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) A^{\pi_\theta}(s,a) \right]$$

Where the advantage $A^{\pi_\theta}(s,a)$ is often estimated using the Temporal Difference (TD) error: $A^{\pi_\theta}(s_t,a_t) \approx r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$.

### 2.5 Proximal Policy Optimization (PPO)

PPO is an advanced policy gradient method that strikes a balance between ease of implementation, sample efficiency, and performance. It improves upon earlier methods like TRPO (Trust Region Policy Optimization) by introducing a clipped surrogate objective function that constrains policy updates, preventing them from becoming too large and destabilizing training.

The clipped surrogate objective for PPO is given by:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min(r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t) \right]$$

where:

- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the ratio of the new policy's probability to the old policy's probability for action $a_t$ given state $s_t$.
- $\hat{A}_t$ is the advantage estimate at time $t$.
- $\epsilon$ is a small hyperparameter (e.g., 0.1 or 0.2) that defines the clipping range.

The $\min$ operator takes the minimum of the standard policy gradient objective and a clipped version. This ensures that the policy updates do not deviate too far from the old policy, promoting stable learning while still allowing for significant improvements. PPO typically uses Generalized Advantage Estimation (GAE) for more accurate advantage estimates, and often incorporates an entropy bonus to encourage exploration.


In [ ]:
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import torch
import gymnasium as gym

# Add the parent directory of src to sys.path to enable imports
sys.path.insert(0, str(Path(os.getcwd()).parent))

from src.config import Config
from src.utils import (
    train_reinforce_agent,
    train_reinforce_baseline_agent,
    train_actor_critic_agent,
    train_ppo_agent,
    train_continuous_ppo_agent,
    compare_policy_gradient_variants,
    hyperparameter_sensitivity_analysis,
    curriculum_learning_demo,
    plot_policy_gradient_comparison,
)

# Set plot style
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Ensure visualizations directory exists
os.makedirs("visualizations", exist_ok=True)

print("=" * 60)
print("CA6: Policy Gradient Methods - Environment Setup")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {Config.DEVICE}")
print(f"Random Seed: {Config.SEED}")
print("Environment setup complete!")
print("=" * 60)

# Set random seeds for reproducibility
torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(Config.SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 3. Modular Implementation Structure

The codebase for this assignment is structured modularly to promote clarity, reusability, and scalability. All core logic resides within the `src/` directory, following best practices for production-grade research code.

Here's an overview of the `src/` directory and its components:

- `src/config.py`: Centralized configuration file for all hyperparameters, environment names, and global settings (e.g., random seeds, device selection).
- `src/model.py`: Defines the neural network architectures used for policies (discrete and continuous) and value functions.
- `src/agents.py`: Contains the implementations of various policy gradient agents, including `REINFORCEAgent`, `REINFORCEBaselineAgent`, `ActorCriticAgent`, `PPOAgent`, and `ContinuousPPOAgent`.
- `src/losses.py`: (Currently integrated into agents for simplicity, but can be extracted for more complex loss compositions) Would contain definitions for specific loss functions beyond standard MSE or cross-entropy, especially for custom policy gradient objectives.
- `src/data.py`: (Placeholder for more complex environments/data loading) Would handle environment creation, wrappers, and potentially data loading for offline RL scenarios.
- `src/utils.py`: Contains utility functions for training loops, performance analysis, and visualization. This includes `train_*_agent` functions, comparison functions, and plotting routines.


## 4. Experimentation and Analysis

This section demonstrates the training and evaluation of various policy gradient agents, along with comparative analyses and advanced experiments.


In [ ]:
# 4.1 Training REINFORCE Agent
print("=" * 60)
print("REINFORCE Agent Training")
print("=" * 60)

reinforce_results = train_reinforce_agent(
    env_name=Config.ENV_NAME_DISCRETE, episodes=Config.REINFORCE_EPISODES
)

print("\n✅ REINFORCE training completed")

In [ ]:
# 4.2 Training REINFORCE with Baseline Agent
print("=" * 60)
print("REINFORCE with Baseline Agent Training")
print("=" * 60)

reinforce_baseline_results = train_reinforce_baseline_agent(
    env_name=Config.ENV_NAME_DISCRETE, episodes=Config.REINFORCE_BASELINE_EPISODES
)

print("\n✅ REINFORCE with Baseline training completed")

In [ ]:
# 4.3 Training Actor-Critic Agent
print("=" * 60)
print("Actor-Critic Agent Training")
print("=" * 60)

actor_critic_results = train_actor_critic_agent(
    env_name=Config.ENV_NAME_DISCRETE, episodes=Config.ACTOR_CRITIC_EPISODES
)

print("\n✅ Actor-Critic training completed")

In [ ]:
# 4.4 Training PPO Agent (Discrete Action Space)
print("=" * 60)
print("PPO Agent Training (Discrete)")
print("=" * 60)

ppo_results = train_ppo_agent(
    env_name=Config.ENV_NAME_DISCRETE, episodes=Config.PPO_EPISODES
)

print("\n✅ PPO training (discrete) completed")

In [ ]:
# 4.5 Training Continuous PPO Agent (Continuous Action Space)
print("=" * 60)
print("Continuous PPO Agent Training")
print("=" * 60)

continuous_ppo_results = train_continuous_ppo_agent(
    env_name=Config.ENV_NAME_CONTINUOUS, episodes=Config.CONTINUOUS_PPO_EPISODES
)

print("\n✅ Continuous PPO training completed")

### 4.6 Comparative Analysis of Policy Gradient Variants

This section runs a comparison of the various policy gradient algorithms implemented and visualizes their performance.


In [ ]:
print("=" * 60)
print("Comparing Policy Gradient Variants")
print("=" * 60)

comparison_results = compare_policy_gradient_variants(
    env_name=Config.ENV_NAME_DISCRETE,
    episodes=Config.REINFORCE_EPISODES
    // 2,  # Use fewer episodes for comparison to save time
)

plot_policy_gradient_comparison(
    comparison_results, save_path="visualizations/policy_gradient_comparison.png"
)

print("\n✅ Policy gradient comparison and plots generated")

### 4.7 Hyperparameter Sensitivity Analysis

This experiment investigates how different hyperparameters impact the performance of a policy gradient agent.


In [ ]:
print("=" * 60)
print("Hyperparameter Sensitivity Analysis")
print("=" * 60)

hyperparameter_results = hyperparameter_sensitivity_analysis(
    env_name=Config.ENV_NAME_DISCRETE,
    episodes=Config.REINFORCE_EPISODES // 5,  # Fewer episodes for sensitivity analysis
)

print("\n✅ Hyperparameter sensitivity analysis completed and plots generated")

### 4.8 Curriculum Learning Demonstration

This section demonstrates a basic curriculum learning setup, where an agent learns in progressively more difficult environments.


In [ ]:
print("=" * 60)
print("Curriculum Learning Demonstration")
print("=" * 60)

curriculum_results = curriculum_learning_demo(
    env_name=Config.ENV_NAME_DISCRETE, episodes=Config.REINFORCE_EPISODES // 2
)

print("\n✅ Curriculum learning demonstration completed and plots generated")

## 5. Conclusion and Future Work

### 5.1 Summary of Findings

This assignment has demonstrated the implementation and comparative analysis of various policy gradient methods. Key findings include:

- **REINFORCE** provides a foundational understanding but suffers from high variance.
- **REINFORCE with Baseline** significantly reduces variance and improves training stability.
- **Actor-Critic methods** further enhance performance by leveraging both a policy (actor) and a value function (critic) for more stable advantage estimation.
- **Proximal Policy Optimization (PPO)** emerges as a robust and high-performing algorithm, balancing sample efficiency with stable updates through its clipped objective.
- **Continuous control** is effectively handled by parameterizing policies as Gaussian distributions, as demonstrated by `ContinuousPPOAgent`.
- **Hyperparameter sensitivity analysis** highlights the importance of proper tuning for optimal performance.
- **Curriculum learning** showcases a strategy to accelerate learning in complex tasks by introducing them incrementally.

### 5.2 Future Work

Possible extensions and areas for future research include:

- **Generalized Advantage Estimation (GAE):** Implementing GAE for more sophisticated advantage estimation in actor-critic and PPO methods.
- **Asynchronous Advantage Actor-Critic (A3C):** Exploring asynchronous training setups with multiple agents for improved sample efficiency and wall-clock time.
- **Trust Region Policy Optimization (TRPO):** Implementing TRPO to understand its theoretical guarantees and practical trade-offs against PPO.
- **Off-Policy Policy Gradients:** Investigating algorithms like DDPG, TD3, or SAC for improved sample efficiency in continuous control tasks.
- **Integration with Complex Environments:** Applying the modularized agents to more challenging environments (e.g., MuJoCo, Atari games) to assess scalability and performance.
- **Meta-Reinforcement Learning:** Exploring how these policy gradient foundations can be extended to learn learning algorithms themselves.
- **Distributed Training:** Implementing distributed training paradigms to leverage multiple computational resources for faster and more stable learning.


## References

[1] Sutton, R. S., & Barto, A. G. (2018). _Reinforcement Learning: An Introduction_. MIT press.

[2] Williams, R. J. (1992). Simple statistical gradient-following algorithms for connectionist reinforcement learning. _Machine Learning_, 8(3-4), 229-256.

[3] Mnih, V., Badia, A. P., Babuschkin, O., Fortunato, M., Silver, D., & Kavukcuoglu, K. (2016). Asynchronous methods for deep reinforcement learning. _International Conference on Machine Learning (ICML)_.

[4] Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. _arXiv preprint arXiv:1707.06347_.
